# MOLMIM Deployment Guide
---

This notebook will demonstrate building and running MolMIM NIM . The notebook will walk you through setting up your NVIDIA API Key, pulling and deploying a MolMIM NIM image.


### Setup NGC API Key

As a requirement for this notebook, you must set up the key as enviroment variable `NGC_API_KEY` to pull the NIMs docker images of your choice. If you haven't gotten your key, please visit the NVIDIA NGC API [homepage](https://org.ngc.nvidia.com/setup/api-key) and generate your API Key. Please run the cell below, input your NGC API KEY in the display textbox, and press the enter key on your keyboard.

In [1]:
import os
import getpass

if not os.environ.get("NGC_API_KEY", ""):
    nvapi_key = getpass.getpass("Enter your NGC API key: ")
    os.environ["NGC_API_KEY"] = nvapi_key


Enter your NGC API key:  ········


### Self-Hosted NIMs

Please execute the cell below to ensure that your docker daemon is up and running.

In [3]:
! docker ps

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


**Expected Output (if you have no running containers):**

```python

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

```

### Login to NVCR (NVIDIA Container Registry)

To access a NIM docker image, you must login via `docker login nvcr.io.` This process requires a default username as `--username $oauthtoken` and `--password-stdin` that accepts the value of `$NGC_API_KEY.`

In [4]:
! echo -e "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin

WARNING! Your password will be stored unencrypted in /home/skhajehei/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded


**Expected Output**:
```
WARNING! Your password will be stored unencrypted in /home/yagupta/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
```

### Pull The Image 

The next step is to Pull the docker image. We demonstrate this step by pulling `molmim:1.0.0`.

In [6]:
! docker pull nvcr.io/nim/nvidia/molmim:1.0.0

1.0.0: Pulling from nim/nvidia/molmim
Digest: sha256:7700c5556935a93055bee5367d36acb6d3e55d22fd1ba28503f5447656fa63fa
Status: Image is up to date for nvcr.io/nim/nvidia/molmim:1.0.0
nvcr.io/nim/nvidia/molmim:1.0.0


**Likely output:** (When you have the image pulled already)
```python

1.0.0: Pulling from nim/nvidia/molmim
Digest: sha256:7700c5556935a93055bee5367d36acb6d3e55d22fd1ba28503f5447656fa63fa
Status: Image is up to date for nvcr.io/nim/nvidia/molmim:1.0.0
nvcr.io/nim/nvidia/molmim:1.0.0
```

Let's check the model image by listing available images. *Please note that the `IMAGE ID` may differ from what you see under the expected output below*.

In [7]:
! docker image ls

REPOSITORY                            TAG       IMAGE ID       CREATED         SIZE
nvcr.io/nim/nvidia/molmim             1.0.0     99f113cd778b   11 months ago   31.9GB
nvcr.io/nim/meta/llama3-8b-instruct   1.0.0     3cb29b0d79e6   13 months ago   12.5GB


**Expected Output**:

```python
REPOSITORY                            TAG       IMAGE ID       CREATED        SIZE
nvcr.io/nim/nvidia/molmim             1.0.0     99f113cd778b   11 months ago   31.9GB
```


#### Setting up Cache for the Model Artifacts

The NIMs download a number of files for ensuring the best profiles are selected to achieve max performance on hardware. Set up location for caching the model artifacts as `LOCAL_NIM_CACHE` and export the variable.

In [8]:
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
!echo $LOCAL_NIM_CACHE

/home/skhajehei/.cache/nim


In [9]:
!mkdir -p "$LOCAL_NIM_CACHE"
# !chmod 777 "$LOCAL_NIM_CACHE"

### Launch NIM LLM Microservice

Launch the NIM LLM microservice by executing the docker run command in the cell bellow.

```python
docker run -it --rm -d --gpus all --name=molmim -e NGC_API_KEY  -v '$LOCAL_NIM_CACHE':/home/nvs/.cache/nim  -p 8000:8000  nvcr.io/nim/nvidia/molmim:1.0.0
```

This Docker command launches NIM LLM microservice using the following flags:

- `-it`: Allocates a pseudo-TTY and keeps STDIN open for interactive processes
- `--rm`: Automatically removes the container when it exits
- `-d`: Runs the container in detached mode (in the background)
- `--gpus all`: Allows the container to access all available GPUs
- `--name=molmim`: Names the container "molmim"
- `-e NGC_API_KEY`: Passes the NGC_API_KEY environment variable to the container
- `-v $LOCAL_NIM_CACHE:/opt/nim/.cache`: Mounts the local NIM cache directory to /home/nvs/.cache/nim in the container
- `-p 8000:8000`: Maps port 8000 on the host to port 8000 in the container
- `nvcr.io/nim/nvidia/molmim:1.0.0`: Specifies the Docker image to use



A system can have multiple running proceesses, so it is must to ensure we are not overtaking a port with any running application. The following code finds a unique free port and allots it:

In [10]:
import random
import socket

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

Your have been alloted the available port: 11182


In [12]:
! docker run --rm -d -it --name molmim --gpus all \
    -e CUDA_VISIBLE_DEVICES=0 \
    -e NGC_API_KEY="$NGC_API_KEY" \
    -v "$LOCAL_NIM_CACHE:/home/nvs/.cache/nim" \
    -p 9000:9000 \
    nvcr.io/nim/nvidia/molmim:1.0.0

# In order to ensure, the local NIM container is completely loaded and doesn't remain in pending stage, we instantiate a wait interval
! sleep 60

3c9a8c151d3d2065e54ecf5894fb7c5ca4f673e7d357c1bd6e1e7ce68dead5d4


In [13]:
! docker logs --tail 45 molmim

Copyright (c) 2012-2014 Deepmind Technologies    (Koray Kavukcuoglu)
Copyright (c) 2011-2012 NEC Laboratories America (Koray Kavukcuoglu)
Copyright (c) 2011-2013 NYU                      (Clement Farabet)
Copyright (c) 2006-2010 NEC Laboratories America (Ronan Collobert, Leon Bottou, Iain Melvin, Jason Weston)
Copyright (c) 2006      Idiap Research Institute (Samy Bengio)
Copyright (c) 2001-2004 Idiap Research Institute (Ronan Collobert, Samy Bengio, Johnny Mariethoz)
Copyright (c) 2015      Google Inc.
Copyright (c) 2015      Yangqing Jia
Copyright (c) 2013-2016 The Caffe contributors
All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

NOTE: The SHMEM allocation li

**Expected Output:**
```
Copyright (c) 2012-2014 Deepmind Technologies    (Koray Kavukcuoglu)
Copyright (c) 2011-2012 NEC Laboratories America (Koray Kavukcuoglu)
Copyright (c) 2011-2013 NYU                      (Clement Farabet)
Copyright (c) 2006-2010 NEC Laboratories America (Ronan Collobert, Leon Bottou, Iain Melvin, Jason Weston)
Copyright (c) 2006      Idiap Research Institute (Samy Bengio)
Copyright (c) 2001-2004 Idiap Research Institute (Ronan Collobert, Samy Bengio, Johnny Mariethoz)
Copyright (c) 2015      Google Inc.
Copyright (c) 2015      Yangqing Jia
Copyright (c) 2013-2016 The Caffe contributors
All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

NOTE: The SHMEM allocation limit is set to the default of 64MB.  This may be
   insufficient for PyTorch.  NVIDIA recommends the use of the following flags:
   docker run --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 ...

{"timestamp": "2025-07-08 21:41:15,841", "level": "INFO", "message": "NIM VERSION:"}
1.0.0
{"timestamp": "2025-07-08 21:41:15,841", "level": "INFO", "message": "NIM LICENSE:"}
GOVERNING TERMS: The NIM container is governed by the NVIDIA Software License Agreement (found at https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-software-license-agreement/) the Product-Specific Terms for NVIDIA AI Products (found at https://www.nvidia.com/en-us/agreements/enterprise-software/product-specific-terms-for-ai-products/); and the use of this model is governed by the AI Foundation Models Community License Agreement (found at https://docs.nvidia.com/ai-foundation-models-community-license.pdf).

Included third-party software licenses are documented in the “molmim_third_party.txt” file found in this directory.

{"timestamp": "2025-07-08 21:41:15,843", "level": "INFO", "message": "Using model config profile ID 'MolMIM'"}
{"timestamp": "2025-07-08 21:41:16,295", "level": "INFO", "message": "Initializing NGC SDK client with NGC org 0710134617208508"}
{"timestamp": "2025-07-08 21:41:16,670", "level": "INFO", "message": "Model destination already exists, skipping download: /home/nvs/.cache/nim/models/molmim_v1.3"}
WARNING: no nltk.download => removed due to security vulnerability: https://github.com/advisories/GHSA-cgvx-9447-vcch
[NeMo W 2025-07-08 21:41:23 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:55: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
      warnings.warn(
    
[NeMo W 2025-07-08 21:41:28 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    
[NeMo I 2025-07-08 21:41:28 megatron_hiddens:110] Registered hidden transform sampled_var_cond_gaussian at bionemo.model.core.hiddens_support.SampledVarGaussianHiddenTransform
[NeMo I 2025-07-08 21:41:28 megatron_hiddens:110] Registered hidden transform interp_var_cond_gaussian at bionemo.model.core.hiddens_support.InterpVarGaussianHiddenTransform
Interactive mode selected, using strategy='auto'
INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```